In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import timm
import pandas as pd
from tqdm import tqdm

# --- 1. Configuration ---
PRETRAINED_WEIGHTS_PATH = '/kaggle/input/okmodel/dino_pretrained_epoch_30.pth'
IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '/kaggle/input/dataset-deepweeds/labels'

# --- Parameters (MODIFIED) ---
BATCH_SIZE = 32
EPOCHS = 50
# --- CHANGE 1: Lower the Learning Rate ---
LEARNING_RATE = 1e-5 # Was 0.0001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Custom Dataset Class Definition ---
class LabeledWeedDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.labels_df)
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx]['Filename'])
        label = self.labels_df.iloc[idx]['Label']
        image = Image.open(img_name).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 3. Data Transforms and Loading (with Stronger Augmentation) ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Use the master label.csv to determine the number of classes
MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')
master_df = pd.read_csv(MASTER_CSV)
NUM_CLASSES = len(master_df['Label'].unique())
print(f"Total number of classes determined from label.csv: {NUM_CLASSES}")

# Load the first fold
TRAIN_CSV = os.path.join(LABEL_DIR, 'train_subset0.csv')
VAL_CSV = os.path.join(LABEL_DIR, 'val_subset0.csv')
train_dataset = LabeledWeedDataset(csv_file=TRAIN_CSV, root_dir=IMAGES_DIR, transform=train_transform)
val_dataset = LabeledWeedDataset(csv_file=VAL_CSV, root_dir=IMAGES_DIR, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Fold 0: Found {len(train_dataset)} training images and {len(val_dataset)} validation images.")

# --- 4. Model Setup ---
model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(PRETRAINED_WEIGHTS_PATH), strict=False)
print(f"Loaded pre-trained DINO weights from: {PRETRAINED_WEIGHTS_PATH}")

# --- 5. Loss, Optimizer, and Training Loop (MODIFIED) ---
criterion = nn.CrossEntropyLoss()
# --- CHANGE 2: Add weight_decay to the optimizer ---
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

print("Starting fine-tuning with Mixed Precision...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss: {running_loss / len(train_loader):.4f}")
    scheduler.step()

    # Validation loop
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{EPOCHS}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1} Validation Accuracy: {accuracy:.2f}%')

# --- 6. Save the Final Classifier ---
torch.save(model.state_dict(), '/kaggle/working/finetuned_classifier_fold0.pth')
print("Finished fine-tuning for Fold 0. Final model saved.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import timm
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Configuration ---
# --- IMPORTANT: Update these paths ---
FINETUNED_MODEL_PATH = '/kaggle/working/finetuned_classifier_fold0.pth'
IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '/kaggle/input/dataset-deepweeds/labels'
TEST_CSV = os.path.join(LABEL_DIR, 'test_subset0.csv')
MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')

# --- Parameters ---
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Custom Dataset Class (You need this again) ---
class LabeledWeedDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.labels_df)
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx]['Filename'])
        label = self.labels_df.iloc[idx]['Label']
        image = Image.open(img_name).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 3. Load Data and Model ---
# Use the same transforms as validation (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get the number of classes and class names from the master file
master_df = pd.read_csv(MASTER_CSV)
NUM_CLASSES = len(master_df['Label'].unique())
# Create a mapping from label index to species name for the report
idx_to_class = {row['Label']: row['Species'] for index, row in master_df.drop_duplicates().iterrows()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class)]

# Create the test dataset and dataloader
test_dataset = LabeledWeedDataset(csv_file=TEST_CSV, root_dir=IMAGES_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Initialize model and load the fine-tuned weights
model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(FINETUNED_MODEL_PATH))
model.eval() # Set the model to evaluation mode

# --- 4. Make Predictions ---
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# --- 5. Calculate and Print Metrics ---
# Overall Test Accuracy
test_accuracy = accuracy_score(all_labels, all_preds)
print(f"\nOverall Test Set Accuracy: {test_accuracy * 100:.2f}%")

# Detailed Classification Report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import timm
import pandas as pd
from tqdm import tqdm

# --- 1. Configuration ---
# --- IMPORTANT: Just change the MODEL_NAME for each experiment ---
MODEL_NAME = 'resnest50d' # <-- CHANGE THIS LINE FOR EACH MODEL

# --- CHOOSE FROM YOUR LIST ---
# 'resnest50d'
# 'efficientnet_b3'
# 'resnext50_32x4d'
# 'inception_v3'

IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '/kaggle/input/dataset-deepweeds/labels'

# --- Parameters ---
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4 # Standard LR is fine for ImageNet models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Custom Dataset Class Definition ---
class LabeledWeedDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.labels_df)
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx]['Filename'])
        label = self.labels_df.iloc[idx]['Label']
        image = Image.open(img_name).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 3. Data Transforms and Loading ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')
master_df = pd.read_csv(MASTER_CSV)
NUM_CLASSES = len(master_df['Label'].unique())
print(f"Total number of classes: {NUM_CLASSES}")

TRAIN_CSV = os.path.join(LABEL_DIR, 'train_subset0.csv')
VAL_CSV = os.path.join(LABEL_DIR, 'val_subset0.csv')
train_dataset = LabeledWeedDataset(csv_file=TRAIN_CSV, root_dir=IMAGES_DIR, transform=train_transform)
val_dataset = LabeledWeedDataset(csv_file=VAL_CSV, root_dir=IMAGES_DIR, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Fold 0: Found {len(train_dataset)} training images and {len(val_dataset)} validation images.")

# --- 4. Model Setup (MODIFIED TO BE DYNAMIC) ---
print(f"Loading model: {MODEL_NAME}")
model = timm.create_model(
    MODEL_NAME,
    pretrained=True, # Use standard ImageNet weights
    num_classes=NUM_CLASSES
).to(device)

# --- 5. Loss, Optimizer, and Training Loop ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

print(f"Starting fine-tuning for {MODEL_NAME}...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss: {running_loss / len(train_loader):.4f}")
    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{EPOCHS}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1} Validation Accuracy: {accuracy:.2f}%')

# --- 6. Save the Final Classifier (MODIFIED TO BE DYNAMIC) ---
save_path = f'/kaggle/working/finetuned_{MODEL_NAME}_fold0.pth'
torch.save(model.state_dict(), save_path)
print(f"Finished fine-tuning for {MODEL_NAME}. Final model saved to {save_path}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import timm
import pandas as pd
from tqdm import tqdm

# --- 1. Configuration ---
# --- IMPORTANT: Just change the MODEL_NAME for each experiment ---
MODEL_NAME = 'resnest50d' # <-- CHANGE THIS LINE FOR EACH MODEL

# --- CHOOSE FROM YOUR LIST ---
# 'resnest50d'
# 'efficientnet_b3'
# 'resnext50_32x4d'
# 'inception_v3'
# 'convnext_base'
# 'efficientnetv2_m'

IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '/kaggle/input/dataset-deepweeds/labels'

# --- Parameters ---
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Custom Dataset Class Definition ---
class LabeledWeedDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.labels_df)
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx]['Filename'])
        label = self.labels_df.iloc[idx]['Label']
        image = Image.open(img_name).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# --- NEW: EarlyStopping Helper Class ---
class EarlyStopping:
    """Stops training when validation accuracy has stopped improving."""
    def __init__(self, patience=5, min_delta=0.01):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_accuracy = 0
        self.stop_training = False

    def __call__(self, val_accuracy):
        if val_accuracy > self.best_accuracy + self.min_delta:
            self.best_accuracy = val_accuracy
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                self.stop_training = True

# --- 3. Data Transforms and Loading ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')
master_df = pd.read_csv(MASTER_CSV)
NUM_CLASSES = len(master_df['Label'].unique())
print(f"Total number of classes: {NUM_CLASSES}")

TRAIN_CSV = os.path.join(LABEL_DIR, 'train_subset0.csv')
VAL_CSV = os.path.join(LABEL_DIR, 'val_subset0.csv')
train_dataset = LabeledWeedDataset(csv_file=TRAIN_CSV, root_dir=IMAGES_DIR, transform=train_transform)
val_dataset = LabeledWeedDataset(csv_file=VAL_CSV, root_dir=IMAGES_DIR, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Fold 0: Found {len(train_dataset)} training images and {len(val_dataset)} validation images.")

# --- 4. Model Setup ---
print(f"Loading model: {MODEL_NAME}")
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
).to(device)

# --- 5. Loss, Optimizer, and Training Loop (MODIFIED) ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

# Initialize the early stopper
early_stopper = EarlyStopping(patience=5, min_delta=0.01)

print(f"Starting fine-tuning for {MODEL_NAME}...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss: {running_loss / len(train_loader):.4f}")
    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{EPOCHS}"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1} Validation Accuracy: {accuracy:.2f}%')

    # Call the early stopper
    early_stopper(accuracy)
    
    # Check if we should stop
    if early_stopper.stop_training:
        break

# --- 6. Save the Final Classifier ---
save_path = f'/kaggle/working/finetuned_{MODEL_NAME}_fold0.pth'
torch.save(model.state_dict(), save_path)
print(f"Finished training for {MODEL_NAME}. Final model saved to {save_path}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import timm
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Configuration ---
# --- MODIFIED: Set the model name and path for resnet50d ---
# --- 1. Configuration ---
# --- MODIFIED: Set the correct model name and path ---
MODEL_NAME = 'resnest50d'
# This path points to the file saved in the current session's working directory
FINETUNED_MODEL_PATH = '/kaggle/working/finetuned_resnest50d_fold0.pth' 

IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '//kaggle/input/dataset-deepweeds/labels'
TEST_CSV = os.path.join(LABEL_DIR, 'test_subset0.csv')
MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')

# --- Parameters ---
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Custom Dataset Class ---
class LabeledWeedDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.labels_df)
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx]['Filename'])
        label = self.labels_df.iloc[idx]['Label']
        image = Image.open(img_name).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 3. Load Data and Model ---
test_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

master_df = pd.read_csv(MASTER_CSV)
NUM_CLASSES = len(master_df['Label'].unique())
idx_to_class = {row['Label']: row['Species'] for index, row in master_df.drop_duplicates().iterrows()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class)]

test_dataset = LabeledWeedDataset(csv_file=TEST_CSV, root_dir=IMAGES_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- MODIFIED: Initialize the correct model architecture ---
print(f"Loading fine-tuned model: {MODEL_NAME}")
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(FINETUNED_MODEL_PATH))
model.eval()

# --- 4. Make Predictions ---
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# --- 5. Calculate and Print Metrics ---
test_accuracy = accuracy_score(all_labels, all_preds)
print(f"\nOverall Test Set Accuracy for {MODEL_NAME}: {test_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix for {MODEL_NAME}')
plt.show()

In [ ]:
# --- 0. Installation (Run this cell once) ---
!pip install grad-cam -q

# --- 1. Imports ---
import torch
from torchvision import transforms
from PIL import Image
import timm
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import os

# --- 2. Configuration ---
MODEL_NAME = 'resnest50d'
FINETUNED_MODEL_PATH = '/kaggle/working/finetuned_resnest50d_fold0.pth' 
IMAGES_DIR = '/kaggle/input/dataset-deepweeds/images'
LABEL_DIR = '/kaggle/input/dataset-deepweeds/labels'
MASTER_CSV = os.path.join(LABEL_DIR, 'labels.csv')
NUM_CLASSES = 9
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_paths_to_visualize = [
    os.path.join(IMAGES_DIR, '/kaggle/input/dataset-deepweeds/images/20160928-140314-0.jpg'),
    os.path.join(IMAGES_DIR, '/kaggle/input/dataset-deepweeds/images/20161207-112141-0.jpg'),
    os.path.join(IMAGES_DIR, '/kaggle/input/dataset-deepweeds/images/20161207-142702-0.jpg')
]

# --- 3. Load Model and Class Names ---
print(f"Loading fine-tuned model: {MODEL_NAME}")
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(FINETUNED_MODEL_PATH))
model.eval()

master_df = pd.read_csv(MASTER_CSV)
idx_to_class = {row['Label']: row['Species'] for index, row in master_df.drop_duplicates().iterrows()}

# --- 4. Identify the Target Layer ---
target_layer = model.layer4[-1]

# --- 5. Main Loop to Generate Visualizations ---
for img_path in image_paths_to_visualize:
    if not os.path.exists(img_path):
        print(f"Warning: Image not found at {img_path}. Skipping.")
        continue
        
    print(f"--- Generating Grad-CAM for: {os.path.basename(img_path)} ---")
    
    # a. Prepare the image
    image = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    input_tensor = transform(image).unsqueeze(0).to(device)

    # b. Set up and run Grad-CAM --- MODIFIED ---
    cam = GradCAM(model=model, target_layers=[target_layer]) # Removed use_cuda
    targets = None 
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :]

    # c. Create the visualization
    rgb_img = cv2.imread(img_path, 1)[:, :, ::-1]
    rgb_img = cv2.resize(rgb_img, (224, 224))
    rgb_img_float = np.float32(rgb_img) / 255
    visualization = show_cam_on_image(rgb_img_float, grayscale_cam, use_rgb=True)

    # d. Get the predicted class name
    pred_class_idx = model(input_tensor).argmax(axis=1).item()
    pred_class_name = idx_to_class.get(pred_class_idx, "Unknown")

    # e. Plot side-by-side
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    
    axs[0].imshow(rgb_img)
    axs[0].set_title('Original Image')
    axs[0].axis('off')

    axs[1].imshow(visualization)
    axs[1].set_title(f'Grad-CAM (Prediction: {pred_class_name})')
    axs[1].axis('off')

    plt.tight_layout()
    plt.show()